# Resources used

In [1]:
import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import graphviz
from datetime import datetime

import sklearn
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ShuffleSplit, GridSearchCV, ParameterGrid, LearningCurveDisplay

import notebook
from platform import python_version

from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_absolute_percentage_error as MAPE

print(f"python: v {python_version()}")
print(f"Jupyter Notebook: v {notebook.__version__}")
print(f"numpy: v {np.__version__}")
print(f"pandas: v {pd.__version__}")
print(f"seaborn: v {sns.__version__}")
print(f"graphviz: v {graphviz.__version__}")
print(f"matplotlib: v {matplotlib.__version__}")
print(f"sklearn: v {sklearn.__version__}")

python: v 3.10.7
Jupyter Notebook: v 6.4.12
numpy: v 1.25.2
pandas: v 2.2.3
seaborn: v 0.13.2
graphviz: v 0.20.1
matplotlib: v 3.6.2
sklearn: v 1.4.2


# SOFC data collection

In [35]:
# import data
df = pd.read_excel(f'DATA.xlsx')

# separation of SOFC-data
df = df[['MCGS_TIME',  # datetime of logs
         'V',    # Voltage [V]
         'T16',  # temperature at the SOFC left front point [°C]
         'T17',  # temperature at the SOFC right rear point [°C]
         'T19',  # air temperature at the SOFC inlet [°C]
         'T20',  # hydrogen temperature at the SOFC inlet [°C]
         'T21',  # air temperature at the SOFC outlet [°C]
         'T22',  # hydrogen temperature at the SOFC outlet [°C]
         #'I',    # Current [A]
         #'W',    # Power [Wt]
        ]]

print(df.info())
df.iloc[3430:3440, :]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14427 entries, 0 to 14426
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   MCGS_TIME  14427 non-null  datetime64[ns]
 1   V          14427 non-null  float64       
 2   T16        14427 non-null  float64       
 3   T17        14427 non-null  float64       
 4   T19        14427 non-null  float64       
 5   T20        14427 non-null  float64       
 6   T21        14427 non-null  float64       
 7   T22        14427 non-null  float64       
dtypes: datetime64[ns](1), float64(7)
memory usage: 901.8 KB
None


,MCGS_TIME,V,T16,T17,T19,T20,T21,T22
3430,2024-04-25 00:04:57,41.900002,712.000000,786.700012,623.799988,735.900024,714.200012,652.099976
3431,2024-04-25 00:05:27,41.900002,712.000000,786.700012,623.799988,735.900024,714.200012,652.099976
3432,2024-04-25 00:05:57,41.900002,711.900024,786.700012,623.700012,736.000000,714.200012,652.099976
3433,2024-04-25 00:06:27,42.099998,712.000000,786.700012,623.799988,735.900024,714.299988,652.000000
3434,2024-04-25 00:06:57,41.700001,712.000000,786.799988,623.799988,736.000000,714.299988,652.099976
3435,2024-04-25 00:07:27,41.900002,711.900024,786.700012,623.500000,735.900024,714.099976,652.000000
3436,2024-04-25 00:07:57,41.799999,711.799988,786.700012,623.700012,735.900024,714.099976,652.000000
3437,2024-04-25 00:08:27,42.000000,711.799988,786.599976,623.700012,735.900024,714.099976,652.000000
3438,2024-04-25 00:08:57,41.900002,711.900024,786.700012,623.400024,735.900024,714.099976,652.000000
3439,2024-04-25 00:09:27,41.900002,711.900024,786.700012,623.599976,735.900024,714.099976,652.000000


# Exploratory data analisys

In [36]:
columns = df.columns

# diff calc
for column in df.columns:
    if column == 'MCGS_TIME':
        df['dt'] = df['MCGS_TIME'].diff().dt.total_seconds()
    else:
        df[f'd{column}'] = df[f'{column}'].diff()
        
df.drop(columns=columns, inplace=True)
df

,dt,dV,dT16,dT17,dT19,dT20,dT21,dT22
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30.0,0.000000,0.400009,0.400009,0.299988,0.500000,0.500000,0.399994
2,30.0,0.000000,0.500000,0.399994,0.300018,0.500000,0.399994,0.400009
3,30.0,0.000000,0.500000,0.500000,0.299988,0.500000,0.500000,0.399994
4,30.0,0.000000,0.399994,0.400009,0.299988,0.400024,0.400009,0.400009
...,...,...,...,...,...,...,...,...
14422,30.0,-0.099998,-0.399964,-0.400024,-0.500000,-0.399994,-0.699982,-0.500000
14423,30.0,-0.200001,-0.299988,-0.400024,-0.299988,-0.399994,-0.700012,-0.600006
14424,30.0,-0.300001,-0.200012,-0.500000,-0.600006,-0.400024,-0.700012,-1.799988
14425,30.0,-0.199999,-0.299988,-0.500000,-0.500000,-0.399994,-0.699982,-1.600006


In [37]:
df_dV_dT = pd.DataFrame()
columns = df.columns

# gradient calc
for i in range(2, df.shape[1]):
    df[f'{columns[i]}/{columns[0]}'] = abs(df.iloc[:, i] / df.iloc[:, 0])
    df_dV_dT[f'{columns[1]}/{columns[i]}'] = abs(df.iloc[:, 1] / df.iloc[:, i])
    
df.drop(columns=columns, inplace=True)
df_dV_dT.replace(np.inf, 0, inplace=True)
df

,dT16/dt,dT17/dt,dT19/dt,dT20/dt,dT21/dt,dT22/dt
0,NaN,NaN,NaN,NaN,NaN,NaN
1,0.013334,0.013334,0.010000,0.016667,0.016667,0.013333
2,0.016667,0.013333,0.010001,0.016667,0.013333,0.013334
3,0.016667,0.016667,0.010000,0.016667,0.016667,0.013333
4,0.013333,0.013334,0.010000,0.013334,0.013334,0.013334
...,...,...,...,...,...,...
14422,0.013332,0.013334,0.016667,0.013333,0.023333,0.016667
14423,0.010000,0.013334,0.010000,0.013333,0.023334,0.020000
14424,0.006667,0.016667,0.020000,0.013334,0.023334,0.060000
14425,0.010000,0.016667,0.016667,0.013333,0.023333,0.053334


In [38]:
df_dV_dT

,dV/dT16,dV/dT17,dV/dT19,dV/dT20,dV/dT21,dV/dT22
0,NaN,NaN,NaN,NaN,NaN,NaN
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...
14422,0.250018,0.249980,0.199996,0.249999,0.142858,0.199996
14423,0.666697,0.499973,0.666697,0.500010,0.285711,0.333332
14424,1.499915,0.600002,0.499997,0.749958,0.428566,0.166668
14425,0.666690,0.399998,0.399998,0.500005,0.285720,0.124999


In [39]:
# temperature gradient info
for column in df.columns:
    print(f'{column}:   ', 'max grad:', round(df[f'{column}'].max(), 3), '°C/c',
                           'mean grad:', round(df[f'{column}'].mean(),3), '°C/c', sep='\t')
    
print()

# voltage gradient info
for column in df_dV_dT.columns:
    print(f'{column}:   ', 'max grad:', round(df_dV_dT[f'{column}'].max(), 3), 'mV/°C',
                           'mean grad:', round(df_dV_dT[f'{column}'].mean(),3), 'mV/°C', sep='\t')

dT16/dt:   	max grad:	0.227	°C/c	mean grad:	0.009	°C/c
dT17/dt:   	max grad:	1.09	°C/c	mean grad:	0.01	°C/c
dT19/dt:   	max grad:	1.34	°C/c	mean grad:	0.014	°C/c
dT20/dt:   	max grad:	0.3	°C/c	mean grad:	0.01	°C/c
dT21/dt:   	max grad:	0.303	°C/c	mean grad:	0.009	°C/c
dT22/dt:   	max grad:	0.303	°C/c	mean grad:	0.008	°C/c

dV/dT16:   	max grad:	125.955	mV/°C	mean grad:	0.692	mV/°C
dV/dT17:   	max grad:	134.032	mV/°C	mean grad:	0.655	mV/°C
dV/dT19:   	max grad:	147.947	mV/°C	mean grad:	0.619	mV/°C
dV/dT20:   	max grad:	383.092	mV/°C	mean grad:	0.716	mV/°C
dV/dT21:   	max grad:	180.935	mV/°C	mean grad:	0.751	mV/°C
dV/dT22:   	max grad:	197.047	mV/°C	mean grad:	0.852	mV/°C
